In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import requests
from bs4 import BeautifulSoup
from nltk.tokenize import sent_tokenize
import nltk
import time


In [ ]:
nltk.download('punkt')  # Tải tokenizer nếu chưa có

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
def extract_intro(text, max_sent=3):
    try:
        sents = sent_tokenize(text)
        return " ".join(sents[:max_sent])
    except Exception:
        return ""

In [ ]:
BASE_URL = "https://vnexpress.net"
MAIN_URL = BASE_URL
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": "https://vnexpress.net/",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Cache-Control": "max-age=0",
}


In [ ]:
# B1. Tải trang chính
res = requests.get(MAIN_URL, headers=headers)
soup = BeautifulSoup(res.text, 'html.parser')


In [ ]:
# B2. Tìm tất cả chuyên mục trong <ul class="menu">
# B2. Tìm tất cả chuyên mục trong <ul class="menu">
menu_ul = soup.find("ul", class_="parent")
# menu_div = soup.find_all("div", class_="row-menu")


In [ ]:
print(menu_ul)

<ul class="parent">
<li class="home">
<a class="flexbox" data-medium="Menu-Home" href="/" onclick="trackingLogoHome('logo-header-menu', this.href)" title="Trang chủ">
<svg class="ic ic-home"><use xlink:href="#Home"></use></svg>
<svg class="ic ic-evne"><use xlink:href="#letter-E"></use></svg>
</a>
</li>
<li class="newlest"><a data-medium="Menu-MoiNhat" href="/tin-tuc-24h" title="Mới nhất">Mới nhất</a></li>
<li class="thoisu" data-id="1001005">
<a data-medium="Menu-ThoiSu" href="/thoi-su" title="Thời sự">
Thời sự </a>
</li>
<li class="thegioi" data-id="1001002">
<a data-medium="Menu-TheGioi" href="/the-gioi" title="Thế giới">
Thế giới </a>
</li>
<li class="kinhdoanh" data-id="1003159">
<a data-medium="Menu-KinhDoanh" href="/kinh-doanh" title="Kinh doanh">
Kinh doanh </a>
</li>
<li class="phapluat" data-id="1001007">
<a data-medium="Menu-PhapLuat" href="/phap-luat" title="Pháp luật">
Pháp luật </a>
</li>
<li class="khoahoccongnghe" data-id="1006219">
<a data-medium="Menu-KhoaHocCongNghe" 

In [ ]:
topic_links = [a['href'] for a in menu_ul.find_all("a", href=True)]
topic_links=topic_links[1:]

In [ ]:
import os

In [ ]:
SAVE_DIR = "du_lieu_bao_moi"
os.makedirs(SAVE_DIR, exist_ok=True)

output_path = os.path.join(SAVE_DIR, "true.csv")

# Nếu chưa có file, tạo dòng header
if not os.path.exists(output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        f.write("title,link,summary,intro,label\n")

In [ ]:
# B3. Duyệt từng chuyên mục
i=0
for topic_link in topic_links:
    i+=1
    topic_url = topic_link if topic_link.startswith("http") else BASE_URL + topic_link
    print(f" Chuyên mục {i}: {topic_url}")
# oke
    try:
        topic_res = requests.get(topic_url, headers=headers)
        topic_soup = BeautifulSoup(topic_res.text, 'html.parser')

        # Tìm các bài viết — giả định thẻ <a class="story__title">
        articles = topic_soup.find_all("article", class_="item-news")
        # print(count(articles))
        count = 0

        for article in articles:
            if count >= 2:
                break

            #  Lấy thẻ <a class="cms-link"> trong <h2>
            h3 = article.find("h3", class_="title-news")
            a_tag = h3.find("a") if h3 else None
            if not a_tag or not a_tag.has_attr('href'):
                continue

            link = a_tag['href']
            if not link.startswith("http"):
                link = BASE_URL + link

            title = a_tag.get_text(strip=True)

            print(f" Link bài: {link}")
            print(f" Tiêu đề: {title}")

            #  Mở link và lấy nội dung, giống như các bước trước
            try:
                news_res = requests.get(link, headers=headers)
                res = requests.get(link, headers=headers)
                print("Status code:", res.status_code)
                print("Link thật:", res.url)
                print(res.text[:1000])  # Kiểm tra xem HTML có phải bài viết không
                news_soup = BeautifulSoup(news_res.text, 'html.parser')
                # print(news_soup)
                # # Tóm tắt
                # summary_div = news_soup.find_all("p", class_="description")
                # summary_text = ""
                # if summary_div:
                #     summary_text = " ".join(p.get_text(strip=True) for p in summary_div)

                # # Nội dung chính
                # content_div = news_soup.find("article", class_="fck_detail")  # BỎ dấu cách dư

                # if not content_div:
                #     print("⚠️ Không tìm thấy nội dung chính")
                #     continue

                # paragraphs = content_div.find_all("p")
                # content_text = " ".join(p.get_text(strip=True) for p in paragraphs)

                # sumary_div = news_soup.find_all("p", class_="description")
                # print(f" Tóm tắt: {sumary_div}\n")
                # if not sumary_div:
                #     continue

                # paragraphs =sumary_div
                # summary_text = " ".join(p.get_text(strip=True) for p in paragraphs)


                # # print(f" Tóm tắt: {summary_text}\n")

                # content_div = news_soup.find("article", attrs={"class": "fck_detail"})
                # print(content_div)
                # if not content_div:
                #     continue

                # paragraphs = content_div.find_all("p")

                # content_text = " ".join(p.get_text(strip=True) for p in paragraphs)
                # intro = extract_intro(content_text, max_sent=3)

                # # print(f" 3 câu đầu: {intro}\n")
                # full_text = f"{title}. {summary_text} {intro}"
                # print(full_text)
                # with open(output_path, "a", encoding="utf-8") as f:
                #     f.write(f'"{full_text}"\n')
                count += 1
                time.sleep(1)

            except Exception as e:
                print(f" Lỗi khi đọc bài: {e}")


    except Exception as e:
        print(f"Lỗi khi duyệt chuyên mục: {e}")


 Chuyên mục 1: https://vnexpress.net/tin-tuc-24h
 Link bài: https://vnexpress.net/swiatek-thang-6-0-6-0-o-chung-ket-wimbledon-4913533.html
 Tiêu đề: Swiatek thắng 6-0, 6-0 ở chung kết Wimbledon
Status code: 406
Link thật: https://vnexpress.net/swiatek-thang-6-0-6-0-o-chung-ket-wimbledon-4913533.html
<h1>406 Not Acceptable</h1>
 Link bài: https://vnexpress.net/hung-thuan-cuoc-doi-toi-sang-trang-4913522.html
 Tiêu đề: Hùng Thuận: 'Cuộc đời tôi sang trang'
Status code: 406
Link thật: https://vnexpress.net/hung-thuan-cuoc-doi-toi-sang-trang-4913522.html
<h1>406 Not Acceptable</h1>
 Chuyên mục 2: https://vnexpress.net/thoi-su
 Link bài: https://vnexpress.net/xe-dap-dien-lao-xuong-ho-hai-anh-em-tu-vong-4913502.html
 Tiêu đề: Xe đạp điện lao xuống hồ, hai anh em tử vong
Status code: 406
Link thật: https://vnexpress.net/xe-dap-dien-lao-xuong-ho-hai-anh-em-tu-vong-4913502.html
<h1>406 Not Acceptable</h1>
 Link bài: https://vnexpress.net/thong-xe-ky-thuat-vanh-dai-3-tp-hcm-doan-qua-tay-ninh-truo

KeyboardInterrupt: 

In [ ]:
# !mkdir -p /content/drive/MyDrive/Project_NLP/du_lieu_bao_nhandan/

In [ ]:
# !cp /content/du_lieu_bao_nhandan/true.csv /content/drive/MyDrive/Project_NLP/du_lieu_bao_nhandan/


In [ ]:
!pip install selenium
!apt-get update # cập nhật apt để cài chrome
!apt-get install -y chromium-chromedriver
!cp /usr/lib/chromium-browser/chromedriver /usr/bin

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 499.2/499.2 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.8/129.8 kB 10.2 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.4.0
    Uninstalling urllib3-2.4.0:
      Successfully uninstalled urllib3-2.4.0


Hit:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,757 kB]
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,106 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,103

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

options = Options()
options.add_argument("--headless")  # không hiện cửa sổ
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 ...")  # tương tự như trên

driver = webdriver.Chrome(options=options)

driver.get("https://vnexpress.net/swiatek-thang-6-0-6-0-o-chung-ket-wimbledon-4913533.html")
html = driver.page_source

from bs4 import BeautifulSoup
soup = BeautifulSoup(html, "html.parser")
content = soup.find("article", class_="fck_detail")
print(content.text)

driver.quit()



*Tiếp tục cập nhậtLần đầu sau 37 năm, một trận chung kết đơn ở Grand Slam mới lại kết thúc với cả hai set trắng, kể từ Steffi Graf thắng Natasha Zvereva ở Pháp Mở rộng năm 1988. Cũng như Graf, Iga Swiatek giành danh hiệu Wimbledon đầu tiên với tỷ số 6-0, 6-0. Trận gặp Graf là chung kết Grand Slam đầu tiên, cũng là cuối cùng của Zvereva. Còn với Anisimova, thất bại trước Swiatek cũng là lần đầu cô chơi chung kết giải lớn.

















Iga Swiatek mừng danh hiệu Wimbledon đơn nữ đầu tiên sau trong trận thắng Anisimova ở chung kết trên sân Trung tâm, thành phố London, Vương quốc Anh ngày 12/7/2025. Ảnh: Reuters

AdvertisementADVERTISEMENTPowered by VidcrunchNextStayPlayback speed1x NormalQualityAutoBack360p240p144pAutoBack0.25x0.5x1x Normal1.5x2x/SkipAds by Nếu chỉ tính ở Wimbledon, lần gần nhất chung kết đơn nữ kết thúc 6-0, 6-0 đã cách đây 114 năm, khi Dorothea Lambert Chambers thắng Dora Boothby. Trong kỷ nguyên mở, từ năm 1968, tỷ số này chưa từng xuất hiện trên sân Trung tâm tại

In [ ]:
import requests
from bs4 import BeautifulSoup
import time

BASE_URL = "https://vnexpress.net"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": BASE_URL,
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
}

# Thử cào 1 bài cụ thể
test_url = "https://vnexpress.net/xe-dap-dien-lao-xuong-ho-hai-anh-em-tu-vong-4715530.html"

res = requests.get(test_url, headers=headers)
print("Status:", res.status_code)

if res.status_code == 200:
    soup = BeautifulSoup(res.text, "html.parser")

    title = soup.find("h1", class_="title-detail")
    summary = soup.find("p", class_="description")
    content_div = soup.find("article", class_="fck_detail")

    print("\nTiêu đề:", title.get_text(strip=True) if title else "Không có")
    print("Tóm tắt:", summary.get_text(strip=True) if summary else "Không có")

    if content_div:
        paragraphs = content_div.find_all("p")
        content_text = " ".join(p.get_text(strip=True) for p in paragraphs)
        print("Nội dung:", content_text[:500], "...")
    else:
        print("Không tìm thấy nội dung bài viết")
else:
    print("Request bị chặn hoặc lỗi:", res.status_code)


Status: 406
Request bị chặn hoặc lỗi: 406
